## 00 · Smoke test

Checks the risky pieces before the pipeline depends on them:

* **(a)** `ai_transcribe` on one clip from the Volume (Beta; a workspace admin must turn on the *AI Transcribe* preview)
* **(b)** Gemini audio input through chat completions
* **(c)** Piper TTS installs on serverless and writes a WAV to the Volume
* **(d)** a trace stored in UC accepts an expectation, and `mlflow.genai.evaluate(data=<traces>)` adds feedback to that same trace
* **(e)** the judge model and **(f)** the agent LLM with tool calling

Run `01_setup` first so the clips and voices are in the Volume. **Illustrative data.**

In [ ]:
%pip install -q "mlflow[databricks]>=3.15" openai piper-tts
dbutils.library.restartPython()

In [ ]:
import json, os, sys, time, traceback

sys.path.insert(0, os.path.abspath(".."))
import mlflow
from voice_evals import config as C

results = {"mlflow": mlflow.__version__}
SAMPLE = f"{C.VOLUME_ROOT}/input/u_006.wav"
print(results, SAMPLE, os.path.exists(SAMPLE))

In [ ]:
from voice_evals.candidates import AiTranscribe

try:
    start = time.perf_counter()
    text = AiTranscribe(spark)(SAMPLE)
    results["ai_transcribe"] = {"ok": True, "text": text, "seconds": round(time.perf_counter() - start, 1)}
except Exception as e:
    results["ai_transcribe"] = {"ok": False, "error": str(e)[:500]}
print(results["ai_transcribe"])

In [ ]:
from databricks.sdk import WorkspaceClient
from openai import OpenAI
from voice_evals.candidates import GeminiTranscriber

w = WorkspaceClient()
serving = w.serving_endpoints.get_open_ai_client()
token = w.config.authenticate()["Authorization"].split(" ", 1)[1]
gateway = OpenAI(base_url=f"{w.config.host}/ai-gateway/mlflow/v1", api_key=token)

results["gemini"] = {}
for label, client, model in [
    ("serving-endpoints", serving, "databricks-gemini-3-5-flash"),
    ("ai-gateway", gateway, "system.ai.gemini-3-5-flash"),
    ("serving-endpoints flash-lite", serving, "databricks-gemini-3-1-flash-lite"),
]:
    try:
        results["gemini"][label] = {"ok": True, "model": model, "text": GeminiTranscriber(client, model)(SAMPLE)}
    except Exception as e:
        results["gemini"][label] = {"ok": False, "model": model, "error": str(e)[:300]}
print(json.dumps(results["gemini"], indent=1))

In [ ]:
from voice_evals.audio import download_piper_voice
from voice_evals.candidates import PiperTTS

try:
    voice = "en_US-lessac-medium"
    path = download_piper_voice(C.piper_model_path(voice), C.piper_model_url(voice))
    out = f"{C.VOLUME_ROOT}/tts/smoke/test.wav"
    duration, rate = PiperTTS(path)("The TriLock femoral stem, TLF-2210, lists at $1,840.", out)
    results["piper"] = {"ok": True, "uri": out, "duration_s": duration, "sample_rate_hz": rate}
    results["piper_roundtrip"] = AiTranscribe(spark)(out) if results["ai_transcribe"]["ok"] else None
except Exception:
    results["piper"] = {"ok": False, "error": traceback.format_exc()[-800:]}
print(results["piper"], results.get("piper_roundtrip"))

In [ ]:
from voice_evals.scorers import asr_wer

exp = C.set_experiment()
results["experiment"] = {"id": exp.experiment_id, "trace_location": str(getattr(exp, "trace_location", None))}

with mlflow.start_span(name="voice_turn") as root:
    mlflow.update_current_trace(tags={"voice_run_id": "smoke", "candidate": "smoke", "condition": "as_heard"})
    with mlflow.start_span(name="asr", span_type="ASR") as span:
        span.set_inputs({"audio_uri": SAMPLE})
        span.set_outputs({"text": "what's the list price of tlf 2210"})
    root.set_inputs({"transcript": "what's the list price of tlf 2210", "audio_uri": SAMPLE})
    root.set_outputs({"answer": "It lists at $1,840."})
trace_id = root.trace_id
mlflow.log_expectation(trace_id=trace_id, name="gt_transcript", value="What's the list price of TLF-2210?")

found = []
for _ in range(12):  # UC trace ingestion can lag by a few seconds
    found = [t for t in mlflow.search_traces(locations=[exp.experiment_id], filter_string="tags.voice_run_id = 'smoke'", return_type="list")
             if t.info.trace_id == trace_id and len(t.data.spans) >= 2]
    if found:
        break
    time.sleep(5)
with mlflow.start_run(run_name="eval/smoke"):
    mlflow.genai.evaluate(data=found, scorers=[asr_wer])  # the real Stage 1 scorer
time.sleep(5)
after = mlflow.get_trace(trace_id)
results["uc_trace"] = {
    "trace_id": trace_id,
    "found_by_tag": len(found),
    "assessments": [
        {"name": a.name, "value": str((a.feedback or a.expectation).value) if (a.feedback or a.expectation) else None,
         "error": str(a.error)[:200] if a.error else None}
        for a in after.info.assessments
    ],
}
print(json.dumps(results["uc_trace"], indent=1))

In [ ]:
from mlflow.genai.scorers import Correctness
from voice_evals.candidates import ToolCallingAgent
from voice_evals.tools import default_tools

try:
    fb = Correctness(model=C.JUDGE_MODEL)(
        inputs={"transcript": "What's the list price of TLF-2210?"},
        outputs={"answer": "It lists at $1,840."},
        expectations={"expected_facts": ["TLF-2210 has a list price of $1,840."]},
    )
    results["judge"] = {"ok": True, "value": str(fb.value), "source": str(fb.source), "rationale": (fb.rationale or "")[:200]}
except Exception as e:
    results["judge"] = {"ok": False, "error": str(e)[:300]}

try:
    with mlflow.start_span(name="smoke_agent"):
        mlflow.update_current_trace(tags={"voice_run_id": "smoke"})
        r = ToolCallingAgent(serving, "databricks-gpt-5-4-mini")("What's the list price of TLF-2210?", default_tools())
    results["agent"] = {"ok": True, "answer": r.answer, "tool_calls": r.tool_calls, "latency_ms": r.latency_ms}
except Exception:
    results["agent"] = {"ok": False, "error": traceback.format_exc()[-800:]}
print(json.dumps({k: results[k] for k in ("judge", "agent")}, indent=1, default=str))

In [ ]:
dbutils.notebook.exit(json.dumps(results, default=str))